# 3.2 算子调用通路：一次 FA 调用是怎么进门的

任何昇腾算子被使用时，都有一条固定的「进门动线」：**框架（PyTorch / 推理引擎）→ CANN 算子库（aclnn 接口）→ 算子执行框架 → Tiling 与 Kernel**。本节把这条动线走一遍，并认识 FA 算子的全部核心参数。

![算子调用通路](images/fia_call_path.svg)

## 1. aclnn 两段式接口

CANN 对外以 **aclnn** 前缀暴露算子（如 `aclnnFusedInferAttentionScore`），且一律拆成两个函数：

```cpp
// 第一段：只做检查与 workspace 申请，不真正执行
aclnnStatus aclnnFusedInferAttentionScoreGetWorkspaceSize(
    const aclTensor *query, const aclTensor *key, const aclTensor *value, ...  // 全部输入
    uint64_t *workspaceSize, aclOpExecutor **executor);

// 第二段：真正发起执行
aclnnStatus aclnnFusedInferAttentionScore(
    void *workspace, uint64_t workspaceSize, aclOpExecutor *executor,
    aclrtStream stream);
```

**为什么要拆两段？**

| 拆分点 | 解决的问题 |
|--|--|
| GetWorkspaceSize 先行 | 执行前就必须知道要多大的临时空间（由 **Tiling** 算出），框架好提前统一申请 |
| executor 传递中间产物 | 第一段解析好的输入描述、tiling 结果打包进 `aclOpExecutor`，第二段直接用，避免重复解析 |
| stream 显示传入 | 同一 executor 可在不同流上按序执行 |

这套「检查 / 执行分离」的模式贯穿所有 aclnn 算子接口，FA 只是参数特别多而已。

## 2. FA 算子的核心参数

以推理场景的 FA 单算子为例，kernel 侧拿到的参数列表如下（按类别分组）：

| 类别 | 参数 | 含义 |
|--|--|--|
| 输入张量 | `query` / `key` / `value` | Q、K、V 三个输入张量 |
| 输入张量 | `attenMask` | 注意力掩码（可选），标记哪些位置不参与注意力 |
| 属性 | `numHeads` | 注意力头数 N1（Q 的头数） |
| 属性 | `numKeyValueHeads` | K、V 的头数 N2（GQA 场景下 N2 < N1） |
| 属性 | `scaleValue` | 缩放系数，一般等于 $1/\sqrt{d}$（见 2.3 节） |
| 属性 | `preTokens` / `nextTokens` | Mask 的边界：当前 token 只能关注前 `preTokens` 与后 `nextTokens` 范围内的 token |
| 属性 | `inputLayout` | 辺入数据排布：`BSH` 或 `BNSD` 等（下一节展开） |
| 输出张量 | `attentionOut` | 注意力计算结果 |
| 执行框架 | `workspaceSize` / `executor` | workspace 大小与执行器，承接两段式接口 |

> 对照第 2 章的接口认知：`numHeads` / `numKeyValueHeads` 正是 **GQA** 的入口（N1 个 Q 头共享 N2 组 KV）；`preTokens` / `nextTokens` 是**因果 / 带状掩码**的边界；`inputLayout` 决定搬运方式。**每一个参数都在第 2 章出现过，只是从数学符号变成了工程接口。**

## 3. 单算子直调的完整动线

在算子开发调测阶段，通常不经框架，直接用 aclnn 接口「单算子直调」，动线如下：

```text
① 准备环境
   aclInit → aclrtSetDevice → aclrtCreateStream
        ▼
② 构造输入
   在 Device 上申请 query / key / value / attenMask 内存，
   用 aclCreateTensor 描述 shape、format、dtype
        ▼
③ 第一段接口
   aclnnFusedInferAttentionScoreGetWorkspaceSize(...)
   → 算子执行框架解析输入、触发 Tiling 计算、
     算出 workspaceSize、构造 executor
        ▼
④ 申请 workspace
   aclrtMalloc(workspace, workspaceSize)
        ▼
⑤ 第二段接口
   aclnnFusedInferAttentionScore(workspace, ..., stream)
   → 按 TilingData 里的 blockDim 启动 Kernel，
     任务进入 stream 排队
        ▼
⑥ 同步取回
   aclrtSynchronizeStream → 把 attentionOut 拷回 Host 比对
```

![整体框架](images/fia_code_structure.svg)

**关键认知**：第一段接口（Host 侧）就完成了 **Tiling** 的全部计算——切分参数在 Kernel 启动**之前**就已经算好、序列化进 executor；第二段只是「按下启动按钮」。这就是下一节 Tiling 存在的位置：**它跑在 CPU 上，而不是 NPU 上**。

## 4. 框架集成视角

生产环境中 FA 不被直接调用，而是藏在框架的计算图里：

| 场景 | 调用链 |
|--|--|
| PyTorch 训练/推理 | `torch.nn.functional.scaled_dot_product_attention` → torch-npu 适配层 → aclnn FA 接口 |
| 推理引擎 | 图优化把 Attention 子图折叠成单个 FA 算子节点 → GE 图引擎调度 → 算子执行框架 |

无论哪条链，到 aclnn 接口这一层就汇合了。**做算子开发时，单算子直调是标准调测姿势**——它剥离了框架层的不确定性，让问题定位停在算子本身（第 4 章将用它把官方 FA 算子跑起来）。

## 5. 本节小结

- aclnn 接口两段式：**GetWorkspaceSize 做检查 + Tiling，执行接口按下启动键**。
- FA 参数虽多，但每个都能映射回第 2 章的一个概念：头数 ↔ GQA、preTokens/nextTokens ↔ 掩码边界、scaleValue ↔ Softmax 稳定性。
- Tiling 跑在 Host（CPU）上、Kernel 启动之前；算子真正的三层结构已经在这条动线里现出轮廓。

## 小测验

1. aclnn 接口为什么拆成 `GetWorkspaceSize` 和执行两段？workspace 的大小是由谁、在什么时机算出来的？
2. `numHeads = 32`、`numKeyValueHeads = 8`，这是哪种注意力变体？组数 G 是多少？
3. `preTokens = 131072`、`nextTokens = 0`（其他输入不变），对应第 2 章的哪种掩码场景？